# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [3]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country' ,'disco' ,'hiphop', 'jazz', 'metal','pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = {'bass': 'bass.wav', 'drums': 'drums.wav', 'other': 'other.wav', 'vocals': 'vocals.wav'} # Write here stems file name
STEM_KEYS = ['bass', 'drums', 'other', 'vocals']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    corrupted_count = 0
    small_count = 0
    big_count = 0

    # Iterate through genres
    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue  # skip if folder missing

        valid_songs = []
        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)
            if not os.path.isdir(song_path):
                continue

            # CHECK : File sizes
            sizes = [os.path.getsize(os.path.join(song_path, f)) for f in STEMS.values()]

            if any(size < (5.0491 * 1024 * 1024) for size in sizes):
                small_count += 1

            if any(size > (5.0493 * 1024 * 1024) for size in sizes):
                big_count += 1

            # CHECK : Completeness (Does it have all stems?)
            if not all(os.path.isfile(os.path.join(song_path, f)) for f in STEMS.values()):
                continue

            if any(size < 4096 for size in sizes):
                corrupted_count += 1
                continue  # skip corrupted

            valid_songs.append(song_path)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs, val_songs = valid_songs[:split_idx], valid_songs[split_idx:]

        # Helper function to populate dict
        def add_to_dict(target_dict, genre, song_list):
            for song in song_list:
                for stem_key, stem_file in STEMS.items():
                    target_dict[genre][stem_key].append(os.path.join(song, stem_file))

        # Populate dicts
        add_to_dict(train_dataset, genre, train_songs)
        add_to_dict(val_dataset, genre, val_songs)

    print("Total corrupted (<4KB) + small (<5.0491MB):", corrupted_count + small_count, corrupted_count, small_count)
    print("Absolute difference (big >5.0493MB vs small <5.0491MB):", abs(big_count - small_count))

    return train_dataset, val_dataset


# Example usage
tr, val = build_dataset(DATA_ROOT)

# Count training reggae drum samples
train_reggae_drums = len(tr["reggae"]["drums"])

# Count validation country vocal samples
val_country_vocals = len(val["country"]["vocals"])

# Absolute difference
difference = abs(train_reggae_drums - val_country_vocals)

print("Training reggae drum samples:", train_reggae_drums)
print("Validation country vocal samples:", val_country_vocals)
print("Absolute Difference:", difference)


Total corrupted (<4KB) + small (<5.0491MB): 314 0 314
Absolute difference (big >5.0493MB vs small <5.0491MB): 268
Training reggae drum samples: 83
Validation country vocal samples: 17
Absolute Difference: 66


In [5]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    total_files = sum(len(paths) for genre in dataset_dict.values() for paths in genre.values())    # ---- COUNT TOTAL FILES ----



        # Load Audio

        # Find Non-Silent Intervals

        # CASE A: Fully silent
        # CASE B: START silence
        # CASE C: END silence
        # CASE D: MIDDLE silence

        # Store result
        # if max_silence >= threshold_sec:
        #     records.append({
        #         "Genre": genre,
        #         "Stem": stem_name,
        #         "Duration": round(total_duration, 2),
        #         "Max_Silence_Sec": round(max_silence, 2),
        #         "Silence_Location": ", ".join(silence_type),
        #         "File_Path": file_path
        #     })
    #-------------------------------------------------------------------------
    print(f"Total files to check: {total_files}")

    for genre, stems in dataset_dict.items():
        for stem_name, file_paths in stems.items():
            for file_path in file_paths:
                try:
                    # Load audio
                    y, sr = librosa.load(file_path, sr=sr)

                    # Find non-silent intervals
                    non_silent = librosa.effects.split(y, top_db=top_db)

                    total_duration = librosa.get_duration(y=y, sr=sr)

                    # If no non-silent intervals → fully silent
                    if len(non_silent) == 0:
                        max_silence = total_duration
                        silence_type = ["Fully Silent"]
                    else:
                        # Calculate silence durations between intervals
                        silence_durations = []
                        silence_type = []

                        # Start silence
                        if non_silent[0][0] > 0:
                            silence_durations.append(non_silent[0][0] / sr)
                            silence_type.append("Start")

                        # Middle silences
                        for i in range(1, len(non_silent)):
                            gap = (non_silent[i][0] - non_silent[i-1][1]) / sr
                            if gap > 0:
                                silence_durations.append(gap)
                                silence_type.append("Middle")

                        # End silence
                        if non_silent[-1][1] < len(y):
                            silence_durations.append((len(y) - non_silent[-1][1]) / sr)
                            silence_type.append("End")

                        max_silence = max(silence_durations) if silence_durations else 0

                    # Store result if silence exceeds threshold
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })

                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# Assuming you already ran:
df_silence = find_long_silences(tr, threshold_sec=5, top_db=TOP_DB)

# Total number of files with silence >= 5 seconds
total_silence_files = len(df_silence)
print("Total files with silence >= 5s:", total_silence_files)

# Filter only vocals from df_silence
vocals_silence = df_silence[df_silence["Stem"] == "vocals"]

# Count how many vocal tracks have silence >= 5 seconds
total_vocals_silence = len(vocals_silence)

print("Total vocal tracks with silence >= 5s:", total_vocals_silence)

# Filter only vocals
vocals_silence = df_silence[df_silence["Stem"] == "vocals"]

# Compute average silence length in seconds
avg_silence_vocals = vocals_silence["Max_Silence_Sec"].mean()

print("Average Silence Length in Vocals (secs):", round(avg_silence_vocals, 2))

# Filter for jazz + drums
jazz_drums_silence = df_silence[
    (df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums")
]

# Count how many jazz drum tracks have silence >= 5 seconds
total_jazz_drums_silence = len(jazz_drums_silence)

print("Total jazz drum tracks with silence >= 5s:", total_jazz_drums_silence)

# Filter for jazz + drums + silence location = only middle
jazz_drums_middle_silence = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "Middle")
]

# Count how many tracks meet the condition
total_jazz_drums_middle_silence = len(jazz_drums_middle_silence)

print("Total jazz drum tracks with silence >= 5s and only middle:", total_jazz_drums_middle_silence)

# Filter for jazz + drums + silence >= 5s + max silence >= 10s
jazz_drums_long_silence = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
]

# Count how many tracks meet the condition
total_jazz_drums_long_silence = len(jazz_drums_long_silence)

print("Total jazz drum tracks with silence >= 5s and Max_Silence_Sec >= 10s:", total_jazz_drums_long_silence)


Total files to check: 3320
Total files to check: 3320
Total files with silence >= 5s: 678
Total vocal tracks with silence >= 5s: 315
Average Silence Length in Vocals (secs): 12.78
Total jazz drum tracks with silence >= 5s: 20
Total jazz drum tracks with silence >= 5s and only middle: 0
Total jazz drum tracks with silence >= 5s and Max_Silence_Sec >= 10s: 7


In [6]:
# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
#-------------------------------------------------------------------------
# Hint: Create a pivot Table: Count by Genre vs Stem
pivot = df_silence.pivot_table(index="Genre", columns="Stem", values="File_Path", aggfunc="count", fill_value=0)
print(pivot)


Stem       bass  drums  other  vocals
Genre                                
blues        17     25      6      45
classical    66     57      4      70
country      11     14      1      17
disco         7      2      3      18
hiphop       20      3     17       4
jazz         26     20      1      73
metal         6      2      1      42
pop          10      4      2       5
reggae        6      4      7      14
rock         12      8      1      27


In [7]:
import librosa
import numpy as np

# Select the first song from 'rock'
genre = "rock"
song_index = 0
song_path = os.path.join(DATA_ROOT, genre, os.listdir(os.path.join(DATA_ROOT, genre))[song_index])

# Load all stems
stems = []
for stem_key in STEM_KEYS:
    file_path = os.path.join(song_path, STEMS[stem_key])
    y, sr = librosa.load(file_path, sr=None)
    stems.append(y)

# Align stems by length (pad/truncate to same size)
min_len = min(len(s) for s in stems)
stems = [s[:min_len] for s in stems]

# Combine stems into a mix
mix = np.sum(stems, axis=0)

# Get duration of the mix sample
mix_duration = librosa.get_duration(y=mix, sr=sr)
print("Length of mix sample (secs):", round(mix_duration, 2))

# Assuming you already created `mix` and have `sr` from the previous step
# Compute RMS amplitude
rms = librosa.feature.rms(y=mix)

# Average RMS across frames
rms_value = np.mean(rms)

print("RMS Amplitude of mix sample:", rms_value)

import numpy as np

# Assuming `mix` is your combined audio signal
# Peak normalization
normalized_mix = mix / np.max(np.abs(mix))

# Find the maximum value of the normalized sample
max_peak_value = np.max(normalized_mix)

print("Max value of peak normalized sample:", max_peak_value)


Length of mix sample (secs): 30.01
RMS Amplitude of mix sample: 0.11140452
Max value of peak normalized sample: 1.0


In [8]:
stems_audio = []
try:
    for key in STEM_KEYS:
      pass
    # ------------------- write your code here -------------------------------
    # Load audio (Duration 5.0s for speed/consistency)
    #-------------------------------------------------------------------------
    file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

    # Load audio (limit duration to 5.0s for speed/consistency)
    y, sr = librosa.load(file_path, sr=None)
    stems_audio.append((key, y, sr))

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

mix_duration = librosa.get_duration(y=y, sr=sr) 
print("Length of mix sample (secs):", round(mix_duration, 2))

Audio loaded successfully.
Length of mix sample (secs): 30.01


In [9]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.vstack([y for (_, y, _) in stems_audio])

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw**2))
print(rms_val)

#Peak Normalization
max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------
print(max_val)

0.14412415
1.0
